# SAE Dual-Pipeline Visualization

这个 notebook 读取 `scripts/run_sae_pipeline.py` 生成的 `pipeline_summary.json`，对 `standard` 和 `attnres` 两个模型的 SAE 训练曲线与结构评估结果做并排可视化。

## 1. 定位结果目录

这一步不是可视化本身，而是先自动定位仓库根目录和 `pipeline_summary.json`。

你可以把这里理解成“告诉 notebook 去哪里找刚刚跑完的 SAE 双模型结果”。如果路径不对，后面的图都不会画出来。

In [ ]:
from __future__ import annotations

import csv
import json
import sys
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display

try:
    import pandas as pd
except Exception:
    pd = None

plt.rcParams["axes.unicode_minus"] = False
MODEL_COLORS = {
    "standard": "#1f77b4",
    "attnres": "#ff7f0e",
}


def find_repo_root(start: Path | None = None) -> Path:
    candidate = Path.cwd() if start is None else start
    for path in [candidate, *candidate.parents]:
        if (path / "stream_analysis").exists() and (path / "scripts").exists():
            return path
    raise FileNotFoundError("Could not locate the repo root from the current working directory.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
SUMMARY_PATH = REPO_ROOT / "outputs" / "sae_dual_pipeline" / "pipeline_summary.json"
REPO_ROOT, SUMMARY_PATH

## 2. 加载总结果清单

这一步会读取 `pipeline_summary.json`，然后把两个模型最关键的输出路径和最优验证集重构误差整理成一个表。

这张表的作用是先做一次“结果核对”：确认 `standard` 和 `attnres` 的 checkpoint、SAE 输出目录、评估 summary 都已经生成。

In [ ]:
if not SUMMARY_PATH.is_file():
    raise FileNotFoundError(
        f"Missing pipeline summary: {SUMMARY_PATH}\n"
        "先运行 accelerate launch scripts/run_sae_pipeline.py ... 生成双模型 SAE 结果。"
    )

with SUMMARY_PATH.open("r", encoding="utf-8") as handle:
    pipeline_summary = json.load(handle)

overview_rows = []
for model_type, payload in pipeline_summary["models"].items():
    overview_rows.append(
        {
            "model_type": model_type,
            "checkpoint": payload["checkpoint_path"],
            "best_val_recon_mse": payload["best_val_recon_mse"],
            "sae_dir": payload["sae_dir"],
            "eval_summary_path": payload.get("eval_summary_path", ""),
        }
    )

if pd is not None:
    display(pd.DataFrame(overview_rows))
else:
    overview_rows

## 3. 读取训练曲线和评估 summary

这一步还是准备数据，不会直接画图。它会把每个模型的 `metrics.csv` 和 `eval_summary.json` 读进内存。

后面的所有可视化都依赖这一步输出的 `train_curves` 和 `eval_payloads`。

In [ ]:
def read_metrics_csv(path: str | Path):
    rows = []
    with Path(path).open("r", encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle):
            rows.append({key: float(value) for key, value in row.items()})
    return rows


DECODER_DISPLAY_MAX_LATENTS = None
DECODER_OVERLAP_QUANTILES = (0.01, 0.99)
DECODER_OVERLAP_TOPK_HOTSPOTS = 75
DECODER_SUBMATRIX_TOPK_PAIRS = 100
DECODER_SUBMATRIX_ORDERING_METHOD = "mean_overlap"
DECODER_SUBMATRIX_MAX_TICKS = 20
COACTIVATION_TOPK_PAIRS = 100
COACTIVATION_ORDERING_METHOD = "jaccard_mean"
COACTIVATION_QUANTILES = (0.01, 0.99)
COACTIVATION_MAX_TICKS = 20
COACTIVATION_LIFT_EPS = 1e-12


def load_eval_summary(path: str | Path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def load_decoder_overlap_matrix(checkpoint_path: str | Path):
    payload = torch.load(Path(checkpoint_path), map_location="cpu", weights_only=False)
    w_dec = payload["model_state"]["W_dec"].detach().to(dtype=torch.float32)
    w_dec = w_dec / w_dec.norm(dim=1, keepdim=True).clamp_min(1e-8)
    return (w_dec @ w_dec.T).cpu().numpy()


def maybe_subsample_matrix_for_display(matrix: np.ndarray, max_latents: int | None = DECODER_DISPLAY_MAX_LATENTS):
    if max_latents is None or matrix.shape[0] <= max_latents:
        return matrix, None
    indices = np.linspace(0, matrix.shape[0] - 1, num=max_latents, dtype=int)
    return matrix[np.ix_(indices, indices)], indices


def mask_diagonal_for_display(matrix: np.ndarray) -> np.ndarray:
    masked = np.array(matrix, dtype=float, copy=True)
    np.fill_diagonal(masked, np.nan)
    return masked


def offdiag_values(matrix: np.ndarray) -> np.ndarray:
    if matrix.ndim != 2 or matrix.shape[0] != matrix.shape[1]:
        raise ValueError(f"Expected a square matrix, got {matrix.shape}.")
    mask = ~np.eye(matrix.shape[0], dtype=bool)
    values = np.asarray(matrix[mask], dtype=float)
    return values[np.isfinite(values)]


def compute_robust_color_limits(
    matrix: np.ndarray,
    lower_q: float = DECODER_OVERLAP_QUANTILES[0],
    upper_q: float = DECODER_OVERLAP_QUANTILES[1],
):
    values = offdiag_values(matrix)
    if values.size == 0:
        raise RuntimeError("No finite off-diagonal values found for decoder overlap heatmap.")
    vmin = float(np.quantile(values, lower_q))
    vmax = float(np.quantile(values, upper_q))
    if not np.isfinite(vmin) or not np.isfinite(vmax):
        vmin = float(np.nanmin(values))
        vmax = float(np.nanmax(values))
    if vmin >= vmax:
        center = float(np.nanmedian(values))
        spread = max(float(np.nanstd(values)), 1e-6)
        vmin = center - spread
        vmax = center + spread
    return vmin, vmax


def get_topk_offdiag_pairs(matrix: np.ndarray, topk: int = DECODER_OVERLAP_TOPK_HOTSPOTS):
    if topk <= 0:
        return np.empty((0,), dtype=int), np.empty((0,), dtype=int), np.empty((0,), dtype=float)
    rows, cols = np.triu_indices(matrix.shape[0], k=1)
    values = np.asarray(matrix[rows, cols], dtype=float)
    finite_mask = np.isfinite(values)
    rows = rows[finite_mask]
    cols = cols[finite_mask]
    values = values[finite_mask]
    if values.size == 0:
        return np.empty((0,), dtype=int), np.empty((0,), dtype=int), np.empty((0,), dtype=float)
    topk = min(topk, values.size)
    keep = np.argpartition(values, -topk)[-topk:]
    order = keep[np.argsort(values[keep])[::-1]]
    return rows[order], cols[order], values[order]


def collect_latents_from_pairs(rows: np.ndarray, cols: np.ndarray) -> np.ndarray:
    if rows.size == 0 or cols.size == 0:
        return np.empty((0,), dtype=int)
    return np.unique(np.concatenate([rows, cols]))


def collect_latents_from_pair_records(pair_records) -> np.ndarray:
    if not pair_records:
        return np.empty((0,), dtype=int)
    latents = []
    for row in pair_records:
        latents.extend([int(row["latent_i"]), int(row["latent_j"])])
    return np.unique(np.asarray(latents, dtype=int))


def order_selected_latents(
    submatrix: np.ndarray,
    latent_ids: np.ndarray,
    method: str = DECODER_SUBMATRIX_ORDERING_METHOD,
) -> np.ndarray:
    if latent_ids.size == 0:
        return latent_ids
    if method != "mean_overlap":
        raise ValueError(f"Unsupported ordering method: {method}")
    masked = mask_diagonal_for_display(submatrix)
    scores = np.nanmean(masked, axis=1)
    scores = np.where(np.isfinite(scores), scores, -np.inf)
    order = np.argsort(scores)[::-1]
    return latent_ids[order]


def build_top_pair_induced_submatrix(
    matrix: np.ndarray,
    top_k: int = DECODER_SUBMATRIX_TOPK_PAIRS,
    ordering_method: str = DECODER_SUBMATRIX_ORDERING_METHOD,
):
    pair_rows, pair_cols, pair_values = get_topk_offdiag_pairs(matrix, topk=top_k)
    selected_latents = collect_latents_from_pairs(pair_rows, pair_cols)
    if selected_latents.size == 0:
        raise RuntimeError("No off-diagonal decoder-overlap pairs were selected.")
    unsorted_submatrix = np.asarray(matrix[np.ix_(selected_latents, selected_latents)], dtype=float)
    ordered_latents = order_selected_latents(
        unsorted_submatrix,
        selected_latents,
        method=ordering_method,
    )
    ordered_submatrix = np.asarray(matrix[np.ix_(ordered_latents, ordered_latents)], dtype=float)
    return {
        "pair_rows": pair_rows,
        "pair_cols": pair_cols,
        "pair_values": pair_values,
        "selected_latents": selected_latents,
        "ordered_latents": ordered_latents,
        "ordered_submatrix": ordered_submatrix,
        "ordering_method": ordering_method,
        "top_k": top_k,
    }


def configure_sparse_ticks(axis, size: int, sampled_indices: np.ndarray | None = None, tick_count: int = 6):
    if size <= 1:
        axis.set_xticks([0])
        axis.set_yticks([0])
        return
    ticks = np.linspace(0, size - 1, num=min(tick_count, size), dtype=int)
    axis.set_xticks(ticks)
    axis.set_yticks(ticks)
    labels = sampled_indices[ticks] if sampled_indices is not None else ticks
    axis.set_xticklabels(labels)
    axis.set_yticklabels(labels)


def plot_decoder_overlap_heatmap(
    matrix: np.ndarray,
    *,
    model_type: str,
    sampled_indices: np.ndarray | None = None,
    lower_q: float = DECODER_OVERLAP_QUANTILES[0],
    upper_q: float = DECODER_OVERLAP_QUANTILES[1],
    topk_hotspots: int = DECODER_OVERLAP_TOPK_HOTSPOTS,
):
    display_matrix = np.asarray(matrix, dtype=float)
    masked_matrix = mask_diagonal_for_display(display_matrix)
    vmin, vmax = compute_robust_color_limits(display_matrix, lower_q=lower_q, upper_q=upper_q)
    rows, cols, hotspot_values = get_topk_offdiag_pairs(display_matrix, topk=topk_hotspots)
    offdiag = offdiag_values(display_matrix)

    print(f"[{model_type}] decoder overlap off-diagonal summary")
    print(f"  display_shape: {display_matrix.shape}")
    print(f"  offdiag_mean: {float(np.mean(offdiag)):.6f}")
    print(f"  offdiag_median: {float(np.median(offdiag)):.6f}")
    print(f"  offdiag_max: {float(np.max(offdiag)):.6f}")
    print(f"  robust_vmin: {vmin:.6f}")
    print(f"  robust_vmax: {vmax:.6f}")
    if hotspot_values.size > 0:
        print(f"  topk_hotspots: {hotspot_values.size}")
        print(f"  topk_value_range: [{float(np.min(hotspot_values)):.6f}, {float(np.max(hotspot_values)):.6f}]")
    else:
        print("  topk_hotspots: 0")

    cmap = plt.cm.magma.copy()
    cmap.set_bad(color="#f2f2f2")
    fig, axis = plt.subplots(figsize=(8.8, 7.6), dpi=220)
    image = axis.imshow(
        masked_matrix,
        aspect="equal",
        interpolation="nearest",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        origin="upper",
    )
    if hotspot_values.size > 0:
        overlay_rows = np.concatenate([rows, cols])
        overlay_cols = np.concatenate([cols, rows])
        axis.scatter(
            overlay_cols,
            overlay_rows,
            s=6,
            facecolors="none",
            edgecolors="#7fffd4",
            linewidths=0.25,
            alpha=0.85,
        )
    title = f"{model_type}: decoder overlap heatmap (off-diagonal, robust scale)"
    if sampled_indices is not None:
        title += f" [display sampled to {display_matrix.shape[0]} latents]"
    axis.set_title(title)
    axis.set_xlabel("latent index j")
    axis.set_ylabel("latent index i")
    configure_sparse_ticks(axis, display_matrix.shape[0], sampled_indices=sampled_indices, tick_count=6)
    colorbar = fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    colorbar.set_label("cosine overlap (off-diagonal, robust scale)")
    plt.tight_layout()
    plt.show()


def plot_latent_submatrix_heatmap(
    matrix: np.ndarray,
    *,
    model_type: str,
    top_k: int = DECODER_SUBMATRIX_TOPK_PAIRS,
    ordering_method: str = DECODER_SUBMATRIX_ORDERING_METHOD,
    lower_q: float = DECODER_OVERLAP_QUANTILES[0],
    upper_q: float = DECODER_OVERLAP_QUANTILES[1],
    max_ticks: int = DECODER_SUBMATRIX_MAX_TICKS,
):
    bundle = build_top_pair_induced_submatrix(
        matrix,
        top_k=top_k,
        ordering_method=ordering_method,
    )
    ordered_latents = bundle["ordered_latents"]
    submatrix = bundle["ordered_submatrix"]
    masked_submatrix = mask_diagonal_for_display(submatrix)
    vmin, vmax = compute_robust_color_limits(submatrix, lower_q=lower_q, upper_q=upper_q)
    offdiag = offdiag_values(submatrix)

    print(f"[{model_type}] top-pair-induced submatrix summary")
    print(f"  top_k: {bundle['top_k']}")
    print(f"  selected_latent_count: {ordered_latents.size}")
    print(f"  submatrix_shape: {submatrix.shape}")
    print(f"  ordering_method: {bundle['ordering_method']}")
    print(f"  submatrix_offdiag_mean: {float(np.mean(offdiag)):.6f}")
    print(f"  submatrix_offdiag_max: {float(np.max(offdiag)):.6f}")
    print(f"  robust_vmin: {vmin:.6f}")
    print(f"  robust_vmax: {vmax:.6f}")
    preview_count = min(10, bundle['pair_values'].size)
    print(f"  involved_latent_count: {bundle['selected_latents'].size}")
    for idx in range(preview_count):
        print(
            f"  top_pair_{idx + 1}: ({int(bundle['pair_rows'][idx])}, {int(bundle['pair_cols'][idx])}) -> {float(bundle['pair_values'][idx]):.6f}"
        )

    cmap = plt.cm.magma.copy()
    cmap.set_bad(color="#f2f2f2")
    fig, axis = plt.subplots(figsize=(8.4, 7.4), dpi=220)
    image = axis.imshow(
        masked_submatrix,
        aspect="equal",
        interpolation="nearest",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        origin="upper",
    )
    axis.set_title(
        f"{model_type}: top-pair-induced overlap submatrix (top_k={top_k}, order={ordering_method})"
    )
    axis.set_xlabel("ordered selected latent index j")
    axis.set_ylabel("ordered selected latent index i")
    configure_sparse_ticks(
        axis,
        submatrix.shape[0],
        sampled_indices=ordered_latents,
        tick_count=max_ticks,
    )
    for tick in axis.get_xticklabels():
        tick.set_rotation(90)
    colorbar = fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    colorbar.set_label("cosine overlap (submatrix off-diagonal, robust scale)")
    plt.tight_layout()
    plt.show()


def compute_selected_latent_coactivation_bundle(
    checkpoint_path: str | Path,
    activation_dir: str | Path,
    preprocessing: str,
    selected_latents: np.ndarray,
    *,
    threshold: float = 1e-8,
    batch_size: int = 512,
):
    from torch.utils.data import DataLoader
    from stream_analysis.sae.data import ActivationShardDataset
    from stream_analysis.sae.eval import load_sae_checkpoint

    if selected_latents.size == 0:
        raise RuntimeError("No selected latents were provided for coactivation heatmap.")

    device = torch.device("cpu")
    model, _, _ = load_sae_checkpoint(checkpoint_path, device=device)
    model.eval()
    dataset = ActivationShardDataset(activation_dir, preprocessing=preprocessing)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0, drop_last=False)
    latent_index_tensor = torch.as_tensor(selected_latents, dtype=torch.long)
    pair_counts = torch.zeros((selected_latents.size, selected_latents.size), dtype=torch.float64)
    active_counts = torch.zeros((selected_latents.size,), dtype=torch.float64)
    total_samples = 0

    with torch.no_grad():
        for batch in dataloader:
            if isinstance(batch, (list, tuple)):
                batch = batch[0]
            inputs = batch.to(device=device, dtype=model.b_pre.dtype)
            latents = model(inputs)["z"][:, latent_index_tensor]
            active = (latents.abs() > threshold).to(dtype=torch.float32)
            active_counts += active.sum(dim=0).to(dtype=torch.float64)
            pair_counts += (active.T @ active).to(dtype=torch.float64)
            total_samples += int(active.shape[0])

    if total_samples <= 0:
        raise RuntimeError("No samples were accumulated while recomputing coactivation heatmap.")
    return {
        "pair_counts": pair_counts.cpu().numpy(),
        "active_counts": active_counts.cpu().numpy(),
        "num_samples": float(total_samples),
        "activation_threshold": float(threshold),
    }


def compute_selected_latent_coactivation_matrix(
    checkpoint_path: str | Path,
    activation_dir: str | Path,
    preprocessing: str,
    selected_latents: np.ndarray,
    *,
    threshold: float = 1e-8,
    batch_size: int = 512,
):
    bundle = compute_selected_latent_coactivation_bundle(
        checkpoint_path,
        activation_dir,
        preprocessing,
        selected_latents,
        threshold=threshold,
        batch_size=batch_size,
    )
    return np.asarray(bundle["pair_counts"], dtype=float) / max(float(bundle["num_samples"]), 1.0)


def safe_matrix_divide(numerator: np.ndarray, denominator: np.ndarray) -> np.ndarray:
    numerator = np.asarray(numerator, dtype=float)
    denominator = np.asarray(denominator, dtype=float)
    output = np.zeros_like(numerator, dtype=float)
    np.divide(numerator, denominator, out=output, where=denominator > 0)
    return output


def summarize_offdiag_matrix(matrix: np.ndarray):
    values = offdiag_values(matrix)
    if values.size == 0:
        return {
            "min": float("nan"),
            "mean": float("nan"),
            "median": float("nan"),
            "max": float("nan"),
        }
    return {
        "min": float(np.min(values)),
        "mean": float(np.mean(values)),
        "median": float(np.median(values)),
        "max": float(np.max(values)),
    }


def compute_probability_color_limits(
    matrix: np.ndarray,
    upper_q: float = COACTIVATION_QUANTILES[1],
):
    values = offdiag_values(matrix)
    if values.size == 0:
        return 0.0, 1e-6
    vmax = float(np.quantile(values, upper_q))
    if not np.isfinite(vmax):
        vmax = float(np.nanmax(values))
    if vmax <= 0:
        vmax = max(float(np.nanmax(values)), 1e-6)
    return 0.0, vmax


def compute_diverging_color_limits(
    matrix: np.ndarray,
    lower_q: float = COACTIVATION_QUANTILES[0],
    upper_q: float = COACTIVATION_QUANTILES[1],
):
    values = offdiag_values(matrix)
    if values.size == 0:
        return -1.0, 1.0
    vmin = float(np.quantile(values, lower_q))
    vmax = float(np.quantile(values, upper_q))
    if not np.isfinite(vmin) or not np.isfinite(vmax):
        vmin = float(np.nanmin(values))
        vmax = float(np.nanmax(values))
    if vmin >= vmax:
        center = float(np.nanmedian(values))
        spread = max(float(np.nanstd(values)), 1e-6)
        vmin = center - spread
        vmax = center + spread
    return vmin, vmax


def order_latents_by_normalized_coactivation(
    normalized_matrix: np.ndarray,
    latent_ids: np.ndarray,
    method: str = COACTIVATION_ORDERING_METHOD,
) -> np.ndarray:
    if latent_ids.size == 0:
        return latent_ids
    if method != "jaccard_mean":
        raise ValueError(f"Unsupported coactivation ordering method: {method}")
    masked = mask_diagonal_for_display(normalized_matrix)
    scores = np.nanmean(masked, axis=1)
    scores = np.where(np.isfinite(scores), scores, -np.inf)
    order = np.argsort(scores, kind="stable")[::-1]
    return latent_ids[order]


def resolve_selected_coactivation_latents(
    coactivation_payload: dict,
    top_k_pairs: int = COACTIVATION_TOPK_PAIRS,
):
    all_pair_records = list(coactivation_payload.get("top_coactivated_pairs") or [])
    pair_records = all_pair_records[:top_k_pairs]
    selected_latents = collect_latents_from_pair_records(pair_records)
    selection_source = "top_coactivated_pairs"
    if selected_latents.size == 0:
        sampled_latents = np.asarray(coactivation_payload.get("sampled_latent_indices") or [], dtype=int)
        if sampled_latents.size == 0:
            raise RuntimeError("No coactivation heatmap, top pair records, or sampled latent indices were available.")
        selected_latents = np.unique(sampled_latents[: min(64, sampled_latents.size)])
        selection_source = "sampled_latent_indices"
    return {
        "pair_records": pair_records,
        "selected_latents": selected_latents,
        "selection_source": selection_source,
        "requested_top_k_pairs": int(top_k_pairs),
        "available_pair_count": int(len(all_pair_records)),
        "used_pair_count": int(len(pair_records)),
    }


def build_coactivation_view_bundle(
    eval_payload: dict,
    checkpoint_path: str | Path,
    *,
    top_k_pairs: int = COACTIVATION_TOPK_PAIRS,
    ordering_method: str = COACTIVATION_ORDERING_METHOD,
    threshold: float = 1e-8,
    batch_size: int = 512,
):
    coactivation_payload = eval_payload.get("coactivation", {})
    selection = resolve_selected_coactivation_latents(coactivation_payload, top_k_pairs=top_k_pairs)
    selected_latents = np.asarray(selection["selected_latents"], dtype=int)
    if selected_latents.size < 2:
        raise RuntimeError("Need at least two selected latents to visualize coactivation structure.")

    activation_dir = eval_payload["summary"]["activation_dir"]
    preprocessing = eval_payload["summary"].get("preprocessing", "none")
    stats_bundle = compute_selected_latent_coactivation_bundle(
        checkpoint_path,
        activation_dir,
        preprocessing,
        selected_latents,
        threshold=threshold,
        batch_size=batch_size,
    )
    pair_counts = np.asarray(stats_bundle["pair_counts"], dtype=float)
    active_counts = np.asarray(stats_bundle["active_counts"], dtype=float)
    num_samples = max(float(stats_bundle["num_samples"]), 1.0)

    raw_joint_frequency = pair_counts / num_samples
    active_probability = active_counts / num_samples
    conditional_coactivation = safe_matrix_divide(pair_counts, active_counts[:, None])
    union_counts = active_counts[:, None] + active_counts[None, :] - pair_counts
    jaccard_coactivation = safe_matrix_divide(pair_counts, union_counts)
    expected_under_independence = np.outer(active_probability, active_probability)
    log_lift_coactivation = np.log10(
        (raw_joint_frequency + COACTIVATION_LIFT_EPS) / (expected_under_independence + COACTIVATION_LIFT_EPS)
    )

    ordered_latents = order_latents_by_normalized_coactivation(
        jaccard_coactivation,
        selected_latents,
        method=ordering_method,
    )
    order_lookup = {int(lat): idx for idx, lat in enumerate(selected_latents.tolist())}
    reordered_indices = np.asarray([order_lookup[int(lat)] for lat in ordered_latents], dtype=int)

    def reorder(matrix: np.ndarray) -> np.ndarray:
        array = np.asarray(matrix, dtype=float)
        return np.asarray(array[np.ix_(reordered_indices, reordered_indices)], dtype=float)

    bundle = {
        **selection,
        "ordered_latents": ordered_latents,
        "ordering_method": ordering_method,
        "activation_threshold": float(stats_bundle["activation_threshold"]),
        "num_samples": float(num_samples),
        "raw_joint_frequency": reorder(raw_joint_frequency),
        "conditional_coactivation": reorder(conditional_coactivation),
        "jaccard_coactivation": reorder(jaccard_coactivation),
        "log_lift_coactivation": reorder(log_lift_coactivation),
        "active_probability": np.asarray(active_probability[reordered_indices], dtype=float),
    }
    return bundle


def plot_coactivation_heatmap_view(
    matrix: np.ndarray,
    *,
    title: str,
    colorbar_label: str,
    cmap_name: str,
    scale_mode: str,
    lower_q: float = COACTIVATION_QUANTILES[0],
    upper_q: float = COACTIVATION_QUANTILES[1],
    ordered_latents: np.ndarray | None = None,
    max_ticks: int = COACTIVATION_MAX_TICKS,
):
    display_matrix = np.asarray(matrix, dtype=float)
    masked_matrix = mask_diagonal_for_display(display_matrix)
    norm = None
    if scale_mode == "robust":
        vmin, vmax = compute_robust_color_limits(display_matrix, lower_q=lower_q, upper_q=upper_q)
    elif scale_mode == "probability":
        vmin, vmax = compute_probability_color_limits(display_matrix, upper_q=upper_q)
    elif scale_mode == "diverging":
        vmin, vmax = compute_diverging_color_limits(display_matrix, lower_q=lower_q, upper_q=upper_q)
        if vmin < 0.0 < vmax:
            norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
            vmin = None
            vmax = None
    else:
        raise ValueError(f"Unsupported scale mode: {scale_mode}")

    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad(color="#f2f2f2")
    fig, axis = plt.subplots(figsize=(8.2, 7.2), dpi=220)
    image = axis.imshow(
        masked_matrix,
        aspect="equal",
        interpolation="nearest",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        norm=norm,
        origin="upper",
    )
    axis.set_title(title)
    if ordered_latents is None:
        axis.set_xlabel("latent j")
        axis.set_ylabel("latent i")
        configure_sparse_ticks(axis, display_matrix.shape[0], tick_count=max_ticks)
    else:
        axis.set_xlabel("ordered selected latent index j")
        axis.set_ylabel("ordered selected latent index i")
        configure_sparse_ticks(axis, display_matrix.shape[0], sampled_indices=ordered_latents, tick_count=max_ticks)
    for tick in axis.get_xticklabels():
        tick.set_rotation(90)
    colorbar = fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    colorbar.set_label(colorbar_label)
    plt.tight_layout()
    plt.show()


def plot_raw_coactivation_reference_heatmap(
    matrix: np.ndarray,
    *,
    model_type: str,
    lower_q: float = COACTIVATION_QUANTILES[0],
    upper_q: float = COACTIVATION_QUANTILES[1],
    max_ticks: int = COACTIVATION_MAX_TICKS,
):
    summary = summarize_offdiag_matrix(matrix)
    print(f"[{model_type}] raw coactivation reference heatmap summary")
    print(f"  heatmap_shape: {np.asarray(matrix).shape}")
    print(f"  raw_freq_median: {summary['median']:.6f}")
    print(f"  raw_freq_max: {summary['max']:.6f}")
    plot_coactivation_heatmap_view(
        np.asarray(matrix, dtype=float),
        title=f"{model_type}: raw joint coactivation frequency (eval summary heatmap)",
        colorbar_label="P(i active and j active) [diagonal masked]",
        cmap_name="magma",
        scale_mode="robust",
        lower_q=lower_q,
        upper_q=upper_q,
        ordered_latents=None,
        max_ticks=max_ticks,
    )


def plot_coactivation_submatrix_views(
    eval_payload: dict,
    checkpoint_path: str | Path,
    *,
    model_type: str,
    top_k_pairs: int = COACTIVATION_TOPK_PAIRS,
    ordering_method: str = COACTIVATION_ORDERING_METHOD,
    lower_q: float = COACTIVATION_QUANTILES[0],
    upper_q: float = COACTIVATION_QUANTILES[1],
    max_ticks: int = COACTIVATION_MAX_TICKS,
):
    bundle = build_coactivation_view_bundle(
        eval_payload,
        checkpoint_path,
        top_k_pairs=top_k_pairs,
        ordering_method=ordering_method,
    )
    raw_summary = summarize_offdiag_matrix(bundle["raw_joint_frequency"])
    conditional_summary = summarize_offdiag_matrix(bundle["conditional_coactivation"])
    jaccard_summary = summarize_offdiag_matrix(bundle["jaccard_coactivation"])
    log_lift_summary = summarize_offdiag_matrix(bundle["log_lift_coactivation"])

    print(f"[{model_type}] coactivation selected-latent summary")
    print(f"  selection_source: {bundle['selection_source']}")
    print(f"  requested_top_k_pairs: {bundle['requested_top_k_pairs']}")
    print(f"  available_top_pair_count: {bundle['available_pair_count']}")
    print(f"  used_top_pair_count: {bundle['used_pair_count']}")
    print(f"  selected_latent_count: {bundle['ordered_latents'].size}")
    print(f"  submatrix_shape: {bundle['raw_joint_frequency'].shape}")
    print(f"  ordering_method: {bundle['ordering_method']}")
    print(f"  activation_threshold: {bundle['activation_threshold']:.1e}")
    print(f"  num_samples: {bundle['num_samples']:.0f}")
    print(f"  raw_freq_median: {raw_summary['median']:.6f}")
    print(f"  raw_freq_max: {raw_summary['max']:.6f}")
    print(f"  conditional_median: {conditional_summary['median']:.6f}")
    print(f"  conditional_max: {conditional_summary['max']:.6f}")
    print(f"  jaccard_median: {jaccard_summary['median']:.6f}")
    print(f"  jaccard_max: {jaccard_summary['max']:.6f}")
    print(f"  log_lift_median: {log_lift_summary['median']:.6f}")
    print(f"  log_lift_min: {log_lift_summary['min']:.6f}")
    print(f"  log_lift_max: {log_lift_summary['max']:.6f}")
    preview_count = min(5, bundle['used_pair_count'])
    for idx, row in enumerate(bundle['pair_records'][:preview_count], start=1):
        print(
            f"  top_pair_{idx}: ({int(row['latent_i'])}, {int(row['latent_j'])}) -> {float(row.get('score', float('nan'))):.6f}"
        )

    view_specs = [
        (
            "raw_joint_frequency",
            f"{model_type}: raw joint coactivation frequency (selected latent submatrix)",
            "P(i active and j active) [diagonal masked]",
            "magma",
            "robust",
        ),
        (
            "conditional_coactivation",
            f"{model_type}: conditional coactivation P(j active | i active)",
            "P(j active | i active) [diagonal masked]",
            "viridis",
            "probability",
        ),
        (
            "jaccard_coactivation",
            f"{model_type}: Jaccard coactivation (ordered by jaccard mean)",
            "pairwise Jaccard coactivation [diagonal masked]",
            "viridis",
            "probability",
        ),
        (
            "log_lift_coactivation",
            f"{model_type}: log-lift coactivation vs independence",
            "log10(P(i,j) / (P(i) P(j))) [diagonal masked]",
            "coolwarm",
            "diverging",
        ),
    ]
    for matrix_key, title, colorbar_label, cmap_name, scale_mode in view_specs:
        plot_coactivation_heatmap_view(
            bundle[matrix_key],
            title=title,
            colorbar_label=colorbar_label,
            cmap_name=cmap_name,
            scale_mode=scale_mode,
            lower_q=lower_q,
            upper_q=upper_q,
            ordered_latents=bundle['ordered_latents'],
            max_ticks=max_ticks,
        )
    return bundle


train_curves = {
    model_type: read_metrics_csv(payload["metrics_path"])
    for model_type, payload in pipeline_summary["models"].items()
}
eval_payloads = {
    model_type: load_eval_summary(payload["eval_summary_path"])
    for model_type, payload in pipeline_summary["models"].items()
}

list(train_curves.keys())

## 4. 训练曲线可视化

这张图展示两个模型在 SAE 训练过程中的重构误差变化：

- 左图是 `train_recon_mse`，看训练集上是否稳定下降
- 右图是 `val_recon_mse`，看 held-out 验证激活上的泛化情况

如果右图持续下降且没有明显发散，通常说明这组 SAE 超参数是可用的；如果训练误差很低但验证误差不降，说明可能过拟合或训练数据分布不够稳。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=False)
for axis, metric_key, title in [
    (axes[0], "train_recon_mse", "Train Reconstruction MSE"),
    (axes[1], "val_recon_mse", "Validation Reconstruction MSE"),
]:
    for model_type, rows in train_curves.items():
        steps = [int(row["step"]) for row in rows]
        values = [row[metric_key] for row in rows]
        axis.plot(steps, values, marker="o", label=model_type, color=MODEL_COLORS.get(model_type, "#4c4c4c"))
    axis.set_title(title)
    axis.set_xlabel("step")
    axis.set_ylabel(metric_key)
    axis.grid(alpha=0.3)
    axis.legend()
plt.tight_layout()
plt.show()

## 5. SAE 结构评估指标对比

这一部分会把两个模型最终 SAE 的核心结构指标拆成多张独立柱状图分别比较：

- `recon_mse`：原始重构误差，越低越好
- `normalized_recon_mse`：按输入方差归一化后的重构误差，越低越好
- `avg_l0`：平均激活 latent 数，反映稀疏程度
- `dead_latent_frac`：死特征比例，越低通常越好

这样做的好处是不会把量纲不同的指标挤在同一张图里，更容易单独判断某一个指标上到底是哪边更好。

In [ ]:
metric_specs = [
    ("recon_mse", "Reconstruction MSE", "Lower is better; raw reconstruction error."),
    ("normalized_recon_mse", "Normalized Reconstruction MSE", "Lower is better; normalized by input variance."),
    ("avg_l0", "Average L0", "Average number of active latents per sample."),
    ("dead_latent_frac", "Dead Latent Fraction", "Lower is better; fraction of inactive latents."),
]
model_order = list(pipeline_summary["models"].keys())

for metric_key, title, subtitle in metric_specs:
    values = [float(eval_payloads[model_type]["summary"][metric_key]) for model_type in model_order]
    x = np.arange(len(model_order))
    colors = [MODEL_COLORS.get(model_type, "#4c4c4c") for model_type in model_order]

    fig, ax = plt.subplots(figsize=(7, 4.8))
    bars = ax.bar(x, values, width=0.6, color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels(model_order)
    ax.set_ylabel(metric_key)
    ax.set_title(f"{title}\n{subtitle}")
    ax.grid(axis="y", alpha=0.3)

    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.4f}",
            ha="center",
            va="bottom",
        )

    plt.tight_layout()
    plt.show()

## 6. Decoder Overlap 热图

这张热图展示 SAE decoder 不同 latent 原子之间的余弦相似度。

你可以把它理解成“不同特征之间是不是在重复表示类似方向”。

- 对角线附近或大面积高亮：说明很多 latent 很相似，特征有冗余
- 整体更分散、非对角线不那么亮：说明 decoder 原子更分化

这张图主要是在看特征字典是否“分得开”。

这个优化不改变 overlap 结果本身，只改善显示动态范围和可读性。

主对角线的 self-overlap=1 和大量接近 0 的背景值，会淹没真正重要的 off-diagonal 结构；因此这里会在可视化阶段 mask 主对角线，并用 off-diagonal 分位数做 robust scaling。

图上还会可选叠加 top-k off-diagonal hotspot，帮助你直接看到 overlap 最高的 latent pair。

如果 latent 数特别大，想让图更轻一点，可以把前面代码里的 `DECODER_DISPLAY_MAX_LATENTS` 改成一个整数，比如 `768` 或 `1024`；这只影响显示，不改原矩阵。

In [ ]:
decoder_overlap_matrices = {}

for model_type in model_order:
    checkpoint_path = pipeline_summary["models"][model_type]["sae_best_checkpoint"]
    overlap_matrix = load_decoder_overlap_matrix(checkpoint_path)
    decoder_overlap_matrices[model_type] = overlap_matrix
    display_matrix, sampled_indices = maybe_subsample_matrix_for_display(overlap_matrix)
    plot_decoder_overlap_heatmap(
        display_matrix,
        model_type=model_type,
        sampled_indices=sampled_indices,
    )

## 6.1 Top-Pair-Induced Latent Submatrix Heatmap

这张图不是再画整个全量 overlap 矩阵，而是：

1. 先从 off-diagonal overlap 里找 top-k highest-overlap pairs
2. 收集这些 pair 涉及到的 latent
3. 用这些 latent 构成更小的 overlap 子矩阵
4. 再按 `mean_overlap` 排序后画热力图

这个图更适合回答：高 overlap latent 是不是集中在某个局部小群里，以及这些小群之间有没有可见的 block 结构。

这里同样只改显示方式，不改 overlap 数值本身。

In [ ]:
for model_type in model_order:
    overlap_matrix = decoder_overlap_matrices.get(model_type)
    if overlap_matrix is None:
        checkpoint_path = pipeline_summary["models"][model_type]["sae_best_checkpoint"]
        overlap_matrix = load_decoder_overlap_matrix(checkpoint_path)
        decoder_overlap_matrices[model_type] = overlap_matrix
    plot_latent_submatrix_heatmap(
        overlap_matrix,
        model_type=model_type,
        top_k=DECODER_SUBMATRIX_TOPK_PAIRS,
        ordering_method=DECODER_SUBMATRIX_ORDERING_METHOD,
        max_ticks=DECODER_SUBMATRIX_MAX_TICKS,
    )

## 6.2 Decoder Atom PCA / t-SNE 投影

完整的 `2048×2048` overlap 热图适合看整体分布，但局部结构会非常密。这里补一组二维投影图：

- **PCA**：看 decoder 原子在全局线性主方向上的展开情况
- **t-SNE**：看 decoder 原子在局部邻域结构上是否形成更明显的团簇

这里投影的对象不是 overlap 矩阵本身，而是 **归一化后的 decoder 原子 `W_dec`**。这样更直接，因为 overlap 本来就是这些原子两两余弦相似度的结果。

为了让图更容易读，还会把参与 **top-k overlap hotspot pairs** 的 latent 高亮出来。这样你能同时看：

- 整个字典在二维投影空间里的几何形状
- overlap 最强的 latent 是否集中成一团、形成局部簇，还是分散在不同区域



In [ ]:
try:
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
except Exception as exc:
    raise ImportError("This section requires scikit-learn for PCA / t-SNE.") from exc

DECODER_PROJECTION_MAX_LATENTS = None
DECODER_PROJECTION_TOPK_HOTSPOTS = 75
DECODER_TSNE_PERPLEXITY = 35
DECODER_TSNE_RANDOM_STATE = 42
DECODER_TSNE_METRIC = "cosine"
DECODER_TSNE_PRE_PCA_DIM = 50
DECODER_PROJECTION_POINT_SIZE = 9
DECODER_PROJECTION_HOTSPOT_POINT_SIZE = 20
DECODER_PROJECTION_ANNOTATE_TOPK = 12


def load_decoder_atom_matrix(checkpoint_path: str | Path, normalize: bool = True) -> np.ndarray:
    payload = torch.load(Path(checkpoint_path), map_location="cpu", weights_only=False)
    w_dec = payload["model_state"]["W_dec"].detach().to(dtype=torch.float32).cpu().numpy()
    if normalize:
        norms = np.linalg.norm(w_dec, axis=1, keepdims=True)
        w_dec = w_dec / np.clip(norms, 1e-8, None)
    return np.asarray(w_dec, dtype=np.float32)



def maybe_subsample_decoder_atoms(atom_matrix: np.ndarray, max_latents: int | None = DECODER_PROJECTION_MAX_LATENTS):
    if max_latents is None or atom_matrix.shape[0] <= max_latents:
        indices = np.arange(atom_matrix.shape[0], dtype=int)
        return atom_matrix, indices
    indices = np.linspace(0, atom_matrix.shape[0] - 1, num=max_latents, dtype=int)
    return atom_matrix[indices], indices



def compute_decoder_projection_bundle(
    atom_matrix: np.ndarray,
    *,
    max_latents: int | None = DECODER_PROJECTION_MAX_LATENTS,
    tsne_perplexity: float = DECODER_TSNE_PERPLEXITY,
    tsne_random_state: int = DECODER_TSNE_RANDOM_STATE,
    tsne_metric: str = DECODER_TSNE_METRIC,
    tsne_pre_pca_dim: int = DECODER_TSNE_PRE_PCA_DIM,
):
    sampled_atoms, sampled_indices = maybe_subsample_decoder_atoms(atom_matrix, max_latents=max_latents)
    n_samples, n_features = sampled_atoms.shape
    if n_samples < 3:
        raise RuntimeError(f"Need at least 3 latents for projection, got {n_samples}.")

    pca_2d = PCA(n_components=2)
    pca_coords = pca_2d.fit_transform(sampled_atoms)
    pca_var = pca_2d.explained_variance_ratio_

    pre_pca_dim = min(tsne_pre_pca_dim, n_features, max(2, n_samples - 1))
    if pre_pca_dim >= 2 and pre_pca_dim < n_features:
        tsne_input = PCA(n_components=pre_pca_dim).fit_transform(sampled_atoms)
    else:
        tsne_input = sampled_atoms
        pre_pca_dim = tsne_input.shape[1]

    max_valid_perplexity = max(5, min(float(tsne_perplexity), float(n_samples - 1)))
    if max_valid_perplexity >= n_samples:
        max_valid_perplexity = max(2.0, float(n_samples - 1) / 3.0)
    tsne = TSNE(
        n_components=2,
        perplexity=max_valid_perplexity,
        init="pca",
        learning_rate="auto",
        metric=tsne_metric,
        random_state=tsne_random_state,
    )
    tsne_coords = tsne.fit_transform(tsne_input)

    return {
        "sampled_indices": sampled_indices,
        "pca_coords": np.asarray(pca_coords, dtype=float),
        "pca_explained_variance_ratio": np.asarray(pca_var, dtype=float),
        "tsne_coords": np.asarray(tsne_coords, dtype=float),
        "tsne_perplexity": float(max_valid_perplexity),
        "tsne_metric": tsne_metric,
        "tsne_pre_pca_dim": int(pre_pca_dim),
        "n_samples": int(n_samples),
        "n_features": int(n_features),
    }



def build_hotspot_mask(sampled_indices: np.ndarray, hotspot_latents: np.ndarray) -> np.ndarray:
    hotspot_set = set(int(x) for x in np.asarray(hotspot_latents, dtype=int).tolist())
    return np.asarray([int(idx) in hotspot_set for idx in sampled_indices], dtype=bool)



def annotate_top_latents(axis, coords: np.ndarray, latent_ids: np.ndarray, point_mask: np.ndarray, topk: int = DECODER_PROJECTION_ANNOTATE_TOPK):
    selected_ids = np.asarray(latent_ids[point_mask], dtype=int)
    selected_coords = np.asarray(coords[point_mask], dtype=float)
    if selected_ids.size == 0:
        return
    topk = min(int(topk), selected_ids.size)
    for latent_id, (x, y) in zip(selected_ids[:topk], selected_coords[:topk]):
        axis.text(x, y, str(int(latent_id)), fontsize=7, ha="left", va="bottom")



def plot_single_projection(axis, coords: np.ndarray, sampled_indices: np.ndarray, hotspot_mask: np.ndarray, *, model_type: str, title: str):
    base_color = MODEL_COLORS.get(model_type, "#4c4c4c")
    axis.scatter(
        coords[:, 0],
        coords[:, 1],
        s=DECODER_PROJECTION_POINT_SIZE,
        c=base_color,
        alpha=0.45,
        linewidths=0,
    )
    if np.any(hotspot_mask):
        axis.scatter(
            coords[hotspot_mask, 0],
            coords[hotspot_mask, 1],
            s=DECODER_PROJECTION_HOTSPOT_POINT_SIZE,
            facecolors="none",
            edgecolors="black",
            linewidths=0.65,
        )
        annotate_top_latents(axis, coords, sampled_indices, hotspot_mask)
    axis.set_title(title)
    axis.set_xlabel("dim 1")
    axis.set_ylabel("dim 2")
    axis.grid(alpha=0.2)



def plot_decoder_projection_panel(
    *,
    model_type: str,
    checkpoint_path: str | Path,
    overlap_matrix: np.ndarray,
    max_latents: int | None = DECODER_PROJECTION_MAX_LATENTS,
    topk_hotspots: int = DECODER_PROJECTION_TOPK_HOTSPOTS,
):
    atom_matrix = load_decoder_atom_matrix(checkpoint_path, normalize=True)
    bundle = compute_decoder_projection_bundle(atom_matrix, max_latents=max_latents)
    pair_rows, pair_cols, pair_values = get_topk_offdiag_pairs(overlap_matrix, topk=topk_hotspots)
    hotspot_latents = collect_latents_from_pairs(pair_rows, pair_cols)
    hotspot_mask = build_hotspot_mask(bundle["sampled_indices"], hotspot_latents)

    print(f"[{model_type}] decoder atom projection summary")
    print(f"  normalized_atom_shape: {atom_matrix.shape}")
    print(f"  sampled_latent_count: {bundle['n_samples']}")
    print(f"  hotspot_latent_count: {hotspot_latents.size}")
    print(f"  sampled_hotspot_count: {int(np.sum(hotspot_mask))}")
    print(f"  PCA explained variance ratio: [{bundle['pca_explained_variance_ratio'][0]:.4f}, {bundle['pca_explained_variance_ratio'][1]:.4f}]")
    print(f"  t-SNE perplexity: {bundle['tsne_perplexity']:.1f}")
    print(f"  t-SNE metric: {bundle['tsne_metric']}")
    print(f"  t-SNE pre-PCA dim: {bundle['tsne_pre_pca_dim']}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=220)
    plot_single_projection(
        axes[0],
        bundle["pca_coords"],
        bundle["sampled_indices"],
        hotspot_mask,
        model_type=model_type,
        title=f"{model_type}: decoder atoms PCA (hotspots highlighted)",
    )
    plot_single_projection(
        axes[1],
        bundle["tsne_coords"],
        bundle["sampled_indices"],
        hotspot_mask,
        model_type=model_type,
        title=f"{model_type}: decoder atoms t-SNE (hotspots highlighted)",
    )
    plt.tight_layout()
    plt.show()


for model_type in model_order:
    overlap_matrix = decoder_overlap_matrices.get(model_type)
    if overlap_matrix is None:
        checkpoint_path = pipeline_summary["models"][model_type]["sae_best_checkpoint"]
        overlap_matrix = load_decoder_overlap_matrix(checkpoint_path)
        decoder_overlap_matrices[model_type] = overlap_matrix
    else:
        checkpoint_path = pipeline_summary["models"][model_type]["sae_best_checkpoint"]

    plot_decoder_projection_panel(
        model_type=model_type,
        checkpoint_path=checkpoint_path,
        overlap_matrix=overlap_matrix,
        max_latents=DECODER_PROJECTION_MAX_LATENTS,
        topk_hotspots=DECODER_PROJECTION_TOPK_HOTSPOTS,
    )



## 7. Coactivation 热图

这一部分不再只看 raw joint frequency，而是在同一批 `top_coactivated_pairs` 诱导出的 selected latents 上，同时看 4 个视角：

- raw joint frequency：`P(i active and j active)`
- conditional coactivation：`P(j active | i active)`，它是有方向的，不一定对称
- Jaccard coactivation：`P(i and j) / P(i or j)`，适合看稳定组合
- log-lift：`log10(P(i,j) / (P(i) P(j)))`，适合看是否高于独立性基线

这里会继续保留原始 raw coactivation heatmap，但它不再是唯一主图。因为在 TopK 稀疏 latent 下，raw joint frequency 天然会很小，整张图偏黑不一定表示“没有结构”。

为了让局部组合关系更可读，下面会：

- 默认 mask diagonal，只看 latent 与其他 latent 的关系
- 继续使用 `top_coactivated_pairs` 诱导 selected latent 子集，而不是换数据来源
- 用 Jaccard 关系的 row mean 给 selected latents 排序
- 分别画 raw / conditional / Jaccard / log-lift 四张图

还要注意：decoder overlap 和 coactivation 不是同一种关系。overlap 是 decoder 几何相似性，coactivation 是数据上是否共同激活，所以两者不能直接混为一谈。

In [ ]:
coactivation_view_bundles = {}
for model_type in model_order:
    coactivation_payload = eval_payloads[model_type].get("coactivation", {})
    raw_heatmap = coactivation_payload.get("heatmap")
    if raw_heatmap is not None:
        plot_raw_coactivation_reference_heatmap(
            np.asarray(raw_heatmap, dtype=float),
            model_type=model_type,
            lower_q=COACTIVATION_QUANTILES[0],
            upper_q=COACTIVATION_QUANTILES[1],
            max_ticks=COACTIVATION_MAX_TICKS,
        )

    checkpoint_path = pipeline_summary["models"][model_type]["sae_best_checkpoint"]
    coactivation_view_bundles[model_type] = plot_coactivation_submatrix_views(
        eval_payloads[model_type],
        checkpoint_path,
        model_type=model_type,
        top_k_pairs=COACTIVATION_TOPK_PAIRS,
        ordering_method=COACTIVATION_ORDERING_METHOD,
        lower_q=COACTIVATION_QUANTILES[0],
        upper_q=COACTIVATION_QUANTILES[1],
        max_ticks=COACTIVATION_MAX_TICKS,
    )

list(coactivation_view_bundles.keys())